# Daily Challenge: Building an Agent with LangGraph and the Gemini API

## BaristaBot - A Conversational Cafe Ordering System

This notebook builds a stateful cafe ordering system using **LangGraph** and **Gemini API**.

### What You'll Learn:
- Creating stateful applications with LangGraph
- Integrating Gemini API via LangChain
- Managing state with TypedDict and node functions
- Implementing tool-augmented conversational loops
- Conditional transitions and state management

## Step 1: Install Required Libraries

In [ ]:
#@title Install required packages
%pip install -q "langgraph==1.0.5" "langchain-google-genai==4.1.2" "google-genai==1.56.0"
print("All packages installed successfully!")

## Step 2: Import Required Libraries

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from collections.abc import Iterable
from random import randint
from pprint import pprint
import os

# LangGraph and LangChain imports
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages.ai import AIMessage
from langchain_core.messages.tool import ToolMessage
from langchain_core.tools import tool

print("All imports successful!")

## Step 3: Configure API Key

Add your Google API key to the Colab Secrets (key icon on the left sidebar)

In [ ]:
from google.colab import userdata

# Get API key from Colab Secrets
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("API key configured from Colab Secrets")
except Exception as e:
    print(f"Error: {e}")
    print("Please add GOOGLE_API_KEY to Colab Secrets and run this cell again.")

## Step 4: Define State and System Instructions

In [ ]:
class OrderState(TypedDict):
    """State representing the customer's order conversation."""
    messages: Annotated[list, add_messages]  # Chat conversation
    order: list[str]  # Customer's order items
    finished: bool  # Order completion flag


# System instructions for BaristaBot
BARISTABOT_SYSTEM = (
    "system",
    "You are BaristaBot, an interactive cafe ordering system. You will: "
    "answer questions ONLY about menu items (no off-topic discussion), "
    "help customers place orders for 1 or more items, and use tools to manage "
    "their order (add_to_order, confirm_order, place_order). "
    "IMPORTANT RULES: "
    "1. Always verify drink and modifier names against the MENU before adding to order. "
    "2. Use add_to_order to add items, get_order to view current order, clear_order to reset. "
    "3. Always call confirm_order BEFORE place_order to show the customer their full order. "
    "4. Only use modifiers from the menu - you don't have items not listed. "
    "5. After place_order succeeds, thank the user and say goodbye."
)

WELCOME_MSG = "Welcome to BaristaBot Cafe! Type 'quit' to exit. How can I help you today?"

print("State and instructions defined")

## Step 5: Initialize Gemini Model

In [ ]:
# Initialize Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-latest",
    temperature=0.7,
    max_tokens=1024
)
print("Gemini model initialized")

## Step 6: Define Menu Tool

In [ ]:
@tool
def get_menu() -> str:
    """Retrieve the current cafe menu with all available drinks and modifiers."""
    return """
MENU:

Coffee Drinks:
- Espresso
- Americano
- Cold Brew

Coffee with Milk:
- Latte
- Cappuccino
- Cortado
- Macchiato
- Mocha
- Flat White

Tea Drinks:
- English Breakfast Tea
- Green Tea
- Earl Grey

Tea with Milk:
- Chai Latte
- Matcha Latte
- London Fog

Other Drinks:
- Steamer
- Hot Chocolate

MODIFIERS:
Milk: Whole, 2%, Oat, Almond, 2% Lactose Free (default: Whole)
Espresso shots: Single, Double, Triple, Quadruple (default: Double)
Caffeine: Decaf, Regular (default: Regular)
Temperature: Hot, Iced (default: Hot)
Sweeteners: Vanilla, Hazelnut, Caramel, Chocolate, Sugar-free Vanilla
Special requests: Extra hot, Extra foam, Half caff, etc.

NOTE: Soy milk is currently out of stock.
"""

print("Menu tool defined")

## Step 7: Define Ordering Tools

In [ ]:
@tool
def add_to_order(drink: str, modifiers: Iterable[str] = None) -> str:
    """Add a drink with optional modifiers to the customer's order.
    
    Args:
        drink: The drink name (e.g., 'Latte')
        modifiers: List of modifiers (e.g., ['Oat milk', 'Extra hot'])
    """
    pass


@tool
def get_order() -> str:
    """View the current order. Returns one item per line."""
    pass


@tool
def confirm_order() -> str:
    """Display the complete order to the customer and get their confirmation.
    
    Returns the customer's response (approved, modifications, or cancellation).
    """
    pass


@tool
def clear_order() -> str:
    """Remove all items from the customer's order."""
    pass


@tool
def place_order() -> int:
    """Send the finalized order to the barista for preparation.
    
    Returns:
        Estimated number of minutes until order is ready.
    """
    pass

print("Ordering tools defined")

## Step 8: Define Node Functions

In [ ]:
def chatbot_node(state: OrderState) -> OrderState:
    """Process user input with the Gemini model and tools."""
    if not state.get("messages"):
        # Welcome message on first turn
        response = AIMessage(content=WELCOME_MSG)
    else:
        # Call Gemini with tools
        response = llm_with_tools.invoke(
            [BARISTABOT_SYSTEM] + state["messages"]
        )
    
    return {
        "messages": [response],
        "order": state.get("order", []),
        "finished": state.get("finished", False)
    }


def human_node(state: OrderState) -> OrderState:
    """Display bot message and collect user input."""
    last_message = state["messages"][-1]
    print(f"\nBaristaBot: {last_message.content}")
    print()
    
    user_input = input("You: ").strip()
    
    # Check for quit commands
    if user_input.lower() in {"q", "quit", "exit", "bye", "goodbye"}:
        state["finished"] = True
    
    return state | {"messages": [("user", user_input)]}


def order_node(state: OrderState) -> OrderState:
    """Handle stateful order management (add items, confirm, place)."""
    messages = state.get("messages", [])
    if not messages:
        return state
    
    last_msg = messages[-1]
    if not hasattr(last_msg, "tool_calls") or not last_msg.tool_calls:
        return state
    
    order = state.get("order", [])
    tool_results = []
    finished = state.get("finished", False)
    
    for tool_call in last_msg.tool_calls:
        tool_name = tool_call["name"]
        args = tool_call.get("args", {})
        
        if tool_name == "add_to_order":
            drink = args.get("drink")
            modifiers = args.get("modifiers", [])
            if modifiers:
                item = f"{drink} ({', '.join(modifiers)})"
            else:
                item = drink
            order.append(item)
            response = f"Added: {item}"
        
        elif tool_name == "get_order":
            if order:
                response = "\n".join(f"{i}. {item}" for i, item in enumerate(order, 1))
            else:
                response = "(Empty - no items yet)"
        
        elif tool_name == "confirm_order":
            print("\n" + "="*50)
            print("ORDER CONFIRMATION")
            print("="*50)
            if order:
                for i, item in enumerate(order, 1):
                    print(f"  {i}. {item}")
            else:
                print("  (No items in order)")
            print("="*50 + "\n")
            response = input("Is this correct? (yes/no/changes): ").strip()
        
        elif tool_name == "clear_order":
            order.clear()
            response = "Order cleared"
        
        elif tool_name == "place_order":
            print("\n" + "="*50)
            print("ORDER PLACED!")
            print("="*50)
            print("Items:")
            for i, item in enumerate(order, 1):
                print(f"  {i}. {item}")
            eta = randint(3, 8)
            print(f"Estimated time: {eta} minutes")
            print("="*50 + "\n")
            response = str(eta)
            finished = True
        
        else:
            response = f"Unknown tool: {tool_name}"
        
        tool_results.append(
            ToolMessage(
                content=str(response),
                name=tool_name,
                tool_call_id=tool_call["id"]
            )
        )
    
    return {
        "messages": tool_results,
        "order": order,
        "finished": finished
    }

print("Node functions defined")

## Step 9: Define Routing Functions

In [ ]:
def route_after_human(state: OrderState) -> Literal["chatbot", END]:
    """Route to chatbot or exit based on user action."""
    return END if state.get("finished", False) else "chatbot"


def route_after_chatbot(state: OrderState) -> Literal["tools", "order", "human", END]:
    """Route based on tool calls and completion status."""
    if state.get("finished", False):
        return END
    
    messages = state.get("messages", [])
    if not messages:
        return "human"
    
    last_msg = messages[-1]
    if not hasattr(last_msg, "tool_calls") or not last_msg.tool_calls:
        return "human"
    
    # Check which tools are being called
    auto_tools_names = {"get_menu"}
    has_auto_tool = any(
        call["name"] in auto_tools_names for call in last_msg.tool_calls
    )
    
    if has_auto_tool:
        return "tools"
    else:
        return "order"

print("Routing functions defined")

## Step 10: Build the State Graph

In [ ]:
# Initialize tools
menu_tools = [get_menu]
order_tools_list = [add_to_order, get_order, confirm_order, clear_order, place_order]

# Create tool node for auto tools
auto_tool_node = ToolNode(menu_tools)

# Bind all tools to LLM
llm_with_tools = llm.bind_tools(menu_tools + order_tools_list)

# Build the graph
workflow = StateGraph(OrderState)

# Add nodes
workflow.add_node("chatbot", chatbot_node)
workflow.add_node("human", human_node)
workflow.add_node("tools", auto_tool_node)
workflow.add_node("order", order_node)

# Add edges
workflow.add_edge(START, "chatbot")
workflow.add_conditional_edges("chatbot", route_after_chatbot)
workflow.add_edge("tools", "chatbot")
workflow.add_edge("order", "chatbot")
workflow.add_conditional_edges("human", route_after_human)

# Compile the graph
app = workflow.compile()

print("Graph compiled successfully")

## Step 11: Visualize the Graph (Optional)

In [ ]:
try:
    from IPython.display import Image, display
    
    # Generate and display graph visualization
    graph_png = app.get_graph().draw_mermaid_png()
    display(Image(graph_png))
    print("Graph visualization displayed")
except Exception as e:
    print(f"Could not visualize graph: {e}")
    print("(This is optional - the app will still work)")

## Step 12: Run BaristaBot!

In [ ]:
print("\n" + "="*60)
print(" "*10 + "BARISTABOT CAFE ORDERING SYSTEM")
print("="*60)
print()

config = {"recursion_limit": 100}

try:
    final_state = app.invoke({"messages": []}, config)
    
    print("\n" + "="*60)
    print("Thank you for using BaristaBot! Have a great day!")
    print("="*60)
except KeyboardInterrupt:
    print("\n\nSession interrupted by user.")
except Exception as e:
    print(f"\nError: {e}")
    import traceback
    traceback.print_exc()

## Session Summary (Optional)

In [ ]:
# Display session statistics
print("\nSESSION SUMMARY")
print("-" * 40)
print(f"Order Completed: {final_state.get('finished', False)}")
print(f"Final Order Items: {len(final_state.get('order', []))}")
if final_state.get('order'):
    print("\nItems ordered:")
    for i, item in enumerate(final_state['order'], 1):
        print(f"  {i}. {item}")
print(f"\nTotal Messages: {len(final_state.get('messages', []))}")

## Testing Tips

Try these interactions with BaristaBot:

### Basic Orders
- "I'd like a latte with oat milk"
- "Can I get a cappuccino?"
- "One espresso, please"

### Menu Inquiries
- "What teas do you have?"
- "Show me the menu"
- "What are the milk options?"

### Order Modifications
- "Add an americano to my order"
- "Make the latte extra hot"
- "Can I change the milk to almond?"

### Special Requests
- "No foam, please"
- "Double shots"
- "Sugar-free vanilla sweetener"

### Order Management
- "Show me my current order"
- "Clear my order, let me start over"
- "I'm ready to checkout"

### Exit
- Type: q, quit, exit, bye, or goodbye

### Expected Workflow
1. Bot greets you with a welcome message
2. You describe what you want to order
3. Bot adds items to your order
4. Bot confirms the complete order with you
5. You approve or request changes
6. Bot places the order and provides ETA
7. Bot says goodbye and exits